# 02 — BTS Flight Arrival Data Preparation

## Objective

This notebook prepares scheduled flight arrival data for John F. Kennedy International Airport (JFK) and LaGuardia Airport (LGA) from January to June 2025.

The raw monthly BTS ZIP archives are retained in `data/external/` for reproducibility. This notebook extracts and validates the underlying flight records, restricts the analysis to arrivals at JFK and LGA, and constructs an hourly airport-arrival panel.

The final dataset will use the same airport-hour temporal resolution as the taxi-demand panel produced in `01_taxi_data.ipynb`, allowing the two datasets to be merged in subsequent analysis.

Potential time-lagged relationships between scheduled flight arrivals and taxi pickup demand will be investigated later.

## 1. Setup

We first import the libraries required to inspect the compressed BTS archives and process the flight records. PySpark is used for the main data-processing workflow, while Python's standard library is used to inspect and extract the ZIP archives.

In [3]:
# Core PySpark imports
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

# Python utilities
from pathlib import Path
import zipfile
import os

# Start or retrieve the Spark session
spark = (
    SparkSession.builder
    .appName("MAST30034_Project1_FlightArrivals")
    .getOrCreate()
)

print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/31 18:56:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.2.0


## 2. Locate Raw BTS Flight Archives

The BTS flight data were downloaded as six monthly ZIP archives covering January–June 2025. Before extracting any files, we verify that all expected archives are available.

In [4]:
# Project paths
EXTERNAL_DIR = Path("../data/external")
PROCESSED_DIR = Path("../data/processed")

# Study months
STUDY_MONTHS = [
    "2025_01",
    "2025_02",
    "2025_03",
    "2025_04",
    "2025_05",
    "2025_06",
]

# Expected BTS ZIP archives
flight_zip_paths = [
    EXTERNAL_DIR / f"flight_arrivals_{month}.zip"
    for month in STUDY_MONTHS
]

print("Expected BTS flight archives:")

for path in flight_zip_paths:
    print(f"{path.name}: {'FOUND' if path.exists() else 'MISSING'}")

Expected BTS flight archives:
flight_arrivals_2025_01.zip: FOUND
flight_arrivals_2025_02.zip: FOUND
flight_arrivals_2025_03.zip: FOUND
flight_arrivals_2025_04.zip: FOUND
flight_arrivals_2025_05.zip: FOUND
flight_arrivals_2025_06.zip: FOUND


## 3. Extract Monthly Flight Data

Each BTS archive contains a single `T_ONTIME_REPORTING.csv` file. Since the internal filename is identical across months, each file is extracted and renamed using its corresponding year and month to prevent overwriting.

The original ZIP archives are retained unchanged in `data/external/` to preserve the raw external data and support reproducibility.

In [5]:
# Directory for extracted monthly BTS files
FLIGHT_EXTRACT_DIR = EXTERNAL_DIR / "extracted"

FLIGHT_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

extracted_csv_paths = []

for zip_path, month in zip(flight_zip_paths, STUDY_MONTHS):
    output_path = FLIGHT_EXTRACT_DIR / f"flight_arrivals_{month}.csv"

    with zipfile.ZipFile(zip_path, "r") as zf:
        with zf.open("T_ONTIME_REPORTING.csv") as source:
            with open(output_path, "wb") as target:
                target.write(source.read())

    extracted_csv_paths.append(output_path)

print("Extracted monthly flight files:")

for path in extracted_csv_paths:
    print(path.name)

Extracted monthly flight files:
flight_arrivals_2025_01.csv
flight_arrivals_2025_02.csv
flight_arrivals_2025_03.csv
flight_arrivals_2025_04.csv
flight_arrivals_2025_05.csv
flight_arrivals_2025_06.csv


## 4. Load and Validate BTS Flight Data

The six monthly BTS On-Time Performance files are loaded using PySpark and combined into a single dataset for January--June 2025.

Before constructing hourly flight-arrival measures, we inspect the dataset schema, temporal coverage, airport identifiers, and scheduled arrival-time fields. This ensures that the variables required to align scheduled flight arrivals with hourly taxi pickup demand are available and correctly represented.

In [6]:
# Load all extracted monthly BTS CSV files
flight_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv([str(path) for path in extracted_csv_paths])
)

print("Flight dataset loaded successfully.")
print("Number of rows:", flight_raw.count())
print("Number of columns:", len(flight_raw.columns))

Flight dataset loaded successfully.
Number of rows: 302169
Number of columns: 10


In [7]:
# Inspect the BTS flight dataset schema
flight_raw.printSchema()

root
 |-- FL_DATE: string (nullable = true)
 |-- OP_UNIQUE_CARRIER: string (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- WHEELS_ON: integer (nullable = true)
 |-- CRS_ARR_TIME: integer (nullable = true)
 |-- ARR_TIME: integer (nullable = true)
 |-- ARR_DELAY: double (nullable = true)
 |-- CANCELLED: double (nullable = true)
 |-- DIVERTED: double (nullable = true)



In [8]:
# Display all available column names
for column in flight_raw.columns:
    print(column)

FL_DATE
OP_UNIQUE_CARRIER
ORIGIN
DEST
WHEELS_ON
CRS_ARR_TIME
ARR_TIME
ARR_DELAY
CANCELLED
DIVERTED


In [21]:
# Inspect destination-airport distribution for the airports of interest
flight_raw.filter(
    F.col("DEST").isin(["JFK", "LGA"])
).groupBy("DEST").count().orderBy("DEST").show()

+----+-----+
|DEST|count|
+----+-----+
| JFK|51395|
| LGA|66097|
+----+-----+



In [11]:
# Check missing values in variables required for the analysis
key_flight_cols = [
    "FL_DATE",
    "DEST",
    "CRS_ARR_TIME",
    "ARR_TIME",
    "ARR_DELAY",
    "CANCELLED",
    "DIVERTED"
]

flight_missing_summary = flight_raw.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in key_flight_cols
])

flight_missing_summary.show(truncate=False)

+-------+----+------------+--------+---------+---------+--------+
|FL_DATE|DEST|CRS_ARR_TIME|ARR_TIME|ARR_DELAY|CANCELLED|DIVERTED|
+-------+----+------------+--------+---------+---------+--------+
|0      |0   |0           |5749    |6675     |0        |0       |
+-------+----+------------+--------+---------+---------+--------+



### 4.2 Prepare Scheduled Airport Arrivals

The analysis focuses on flights scheduled to arrive at JFK and LaGuardia. The flight date is first converted from its raw string representation to a proper date type, and the scheduled arrival-time field is validated before constructing hourly arrival timestamps.

Scheduled rather than realised arrival times are used because they represent information that can be known in advance and therefore provide a practical basis for driver positioning decisions.

In [12]:
# Convert the raw flight-date string to a proper date
flight_prepared = flight_raw.withColumn(
    "flight_date",
    F.to_date(
        F.to_timestamp("FL_DATE", "M/d/yyyy h:mm:ss a")
    )
)

flight_prepared.select(
    F.min("flight_date").alias("min_flight_date"),
    F.max("flight_date").alias("max_flight_date")
).show(truncate=False)

+---------------+---------------+
|min_flight_date|max_flight_date|
+---------------+---------------+
|2025-01-01     |2025-06-30     |
+---------------+---------------+



In [13]:
# Inspect scheduled arrival-time values
flight_prepared.select(
    F.min("CRS_ARR_TIME").alias("min_scheduled_arrival"),
    F.max("CRS_ARR_TIME").alias("max_scheduled_arrival"),
    F.sum((F.col("CRS_ARR_TIME") == 2400).cast("int")).alias("time_2400"),
    F.sum((F.col("CRS_ARR_TIME") < 0).cast("int")).alias("time_below_0"),
    F.sum((F.col("CRS_ARR_TIME") > 2400).cast("int")).alias("time_above_2400")
).show(truncate=False)

+---------------------+---------------------+---------+------------+---------------+
|min_scheduled_arrival|max_scheduled_arrival|time_2400|time_below_0|time_above_2400|
+---------------------+---------------------+---------+------------+---------------+
|1                    |2359                 |0        |0           |0              |
+---------------------+---------------------+---------+------------+---------------+



In [14]:
# Check whether the minute component of scheduled arrival times is valid
flight_prepared.select(
    F.sum(
        ((F.col("CRS_ARR_TIME") % 100) >= 60).cast("int")
    ).alias("invalid_minute_values")
).show()

+---------------------+
|invalid_minute_values|
+---------------------+
|                    0|
+---------------------+



### 4.3 Filter and Construct Scheduled Arrivals

The flight data is restricted to flights scheduled to arrive at JFK and LaGuardia during the study period. Cancelled and diverted flights are excluded because they do not represent passengers completing their planned arrival at the destination airport and therefore cannot directly generate taxi pickup demand there.

The scheduled arrival date and time are then combined into an hourly timestamp. This allows flight activity to be aggregated at the same airport-hour resolution as the taxi demand data.

In [22]:
# Inspect cancellation and diversion status for JFK/LGA arrivals
airport_flights = flight_prepared.filter(
    F.col("DEST").isin(["JFK", "LGA"])
)

airport_flights.groupBy(
    "CANCELLED", "DIVERTED"
).count().orderBy(
    "CANCELLED", "DIVERTED"
).show()

+---------+--------+------+
|CANCELLED|DIVERTED| count|
+---------+--------+------+
|      0.0|     0.0|114892|
|      0.0|     1.0|   480|
|      1.0|     0.0|  2120|
+---------+--------+------+



In [23]:
# Retain all flights scheduled to arrive at JFK and LGA.
# Cancellation and diversion outcomes are not used because they may not
# be known at the prediction time and could introduce look-ahead leakage.
scheduled_arrivals = airport_flights

print("Scheduled JFK/LGA arrivals:", scheduled_arrivals.count())

scheduled_arrivals.groupBy("DEST").count().orderBy("DEST").show()

print("JFK/LGA arrivals before filtering:", airport_flights.count())
print("Usable scheduled arrivals:", scheduled_arrivals.count())
print(
    "Records removed:",
    airport_flights.count() - scheduled_arrivals.count()
)

scheduled_arrivals.groupBy("DEST").count().orderBy("DEST").show()

Scheduled JFK/LGA arrivals: 117492
+----+-----+
|DEST|count|
+----+-----+
| JFK|51395|
| LGA|66097|
+----+-----+

JFK/LGA arrivals before filtering: 117492
Usable scheduled arrivals: 117492
Records removed: 0
+----+-----+
|DEST|count|
+----+-----+
| JFK|51395|
| LGA|66097|
+----+-----+



In [24]:
# Construct scheduled arrival hour
scheduled_arrivals = (
    scheduled_arrivals
    .withColumn(
        "scheduled_hour",
        F.floor(F.col("CRS_ARR_TIME") / 100).cast("int")
    )
)

scheduled_arrivals.select(
    "flight_date",
    "DEST",
    "CRS_ARR_TIME",
    "scheduled_hour"
).show(20, truncate=False)

+-----------+----+------------+--------------+
|flight_date|DEST|CRS_ARR_TIME|scheduled_hour|
+-----------+----+------------+--------------+
|2025-05-01 |JFK |1151        |11            |
|2025-05-01 |JFK |2030        |20            |
|2025-05-01 |JFK |1030        |10            |
|2025-05-01 |JFK |1458        |14            |
|2025-05-01 |JFK |1900        |19            |
|2025-05-01 |JFK |2203        |22            |
|2025-05-01 |LGA |904         |9             |
|2025-05-01 |LGA |1105        |11            |
|2025-05-01 |LGA |1203        |12            |
|2025-05-01 |LGA |1259        |12            |
|2025-05-01 |LGA |1505        |15            |
|2025-05-01 |LGA |1620        |16            |
|2025-05-01 |LGA |1819        |18            |
|2025-05-01 |LGA |2005        |20            |
|2025-05-01 |LGA |2230        |22            |
|2025-05-01 |LGA |2343        |23            |
|2025-05-01 |JFK |1329        |13            |
|2025-05-01 |JFK |1730        |17            |
|2025-05-01 |

In [25]:
# Combine flight date and scheduled hour into a timezone-independent
# local hourly timestamp
scheduled_arrivals = scheduled_arrivals.withColumn(
    "arrival_hour",
    F.concat(
        F.date_format("flight_date", "yyyy-MM-dd"),
        F.lit(" "),
        F.lpad(F.col("scheduled_hour").cast("string"), 2, "0"),
        F.lit(":00:00")
    ).cast("timestamp_ntz")
)

scheduled_arrivals.select(
    "DEST",
    "flight_date",
    "CRS_ARR_TIME",
    "arrival_hour"
).show(20, truncate=False)

+----+-----------+------------+-------------------+
|DEST|flight_date|CRS_ARR_TIME|arrival_hour       |
+----+-----------+------------+-------------------+
|JFK |2025-05-01 |1151        |2025-05-01 11:00:00|
|JFK |2025-05-01 |2030        |2025-05-01 20:00:00|
|JFK |2025-05-01 |1030        |2025-05-01 10:00:00|
|JFK |2025-05-01 |1458        |2025-05-01 14:00:00|
|JFK |2025-05-01 |1900        |2025-05-01 19:00:00|
|JFK |2025-05-01 |2203        |2025-05-01 22:00:00|
|LGA |2025-05-01 |904         |2025-05-01 09:00:00|
|LGA |2025-05-01 |1105        |2025-05-01 11:00:00|
|LGA |2025-05-01 |1203        |2025-05-01 12:00:00|
|LGA |2025-05-01 |1259        |2025-05-01 12:00:00|
|LGA |2025-05-01 |1505        |2025-05-01 15:00:00|
|LGA |2025-05-01 |1620        |2025-05-01 16:00:00|
|LGA |2025-05-01 |1819        |2025-05-01 18:00:00|
|LGA |2025-05-01 |2005        |2025-05-01 20:00:00|
|LGA |2025-05-01 |2230        |2025-05-01 22:00:00|
|LGA |2025-05-01 |2343        |2025-05-01 23:00:00|
|JFK |2025-0

In [26]:
# Validate constructed scheduled-arrival timestamps
scheduled_arrivals.select(
    F.min("arrival_hour").alias("min_arrival_hour"),
    F.max("arrival_hour").alias("max_arrival_hour"),
    F.sum(F.col("arrival_hour").isNull().cast("int")).alias("missing_arrival_hour")
).show(truncate=False)

+-------------------+-------------------+--------------------+
|min_arrival_hour   |max_arrival_hour   |missing_arrival_hour|
+-------------------+-------------------+--------------------+
|2025-01-01 00:00:00|2025-06-30 23:00:00|0                   |
+-------------------+-------------------+--------------------+



In [27]:
# Aggregate scheduled arrivals to the airport-hour level
hourly_flight_arrivals = (
    scheduled_arrivals
    .groupBy("DEST", "arrival_hour")
    .agg(
        F.count("*").alias("scheduled_arrivals")
    )
    .withColumnRenamed("DEST", "airport")
    .orderBy("airport", "arrival_hour")
)

hourly_flight_arrivals.show(20, truncate=False)

+-------+-------------------+------------------+
|airport|arrival_hour       |scheduled_arrivals|
+-------+-------------------+------------------+
|JFK    |2025-01-01 00:00:00|4                 |
|JFK    |2025-01-01 04:00:00|1                 |
|JFK    |2025-01-01 05:00:00|5                 |
|JFK    |2025-01-01 06:00:00|10                |
|JFK    |2025-01-01 07:00:00|13                |
|JFK    |2025-01-01 08:00:00|10                |
|JFK    |2025-01-01 09:00:00|9                 |
|JFK    |2025-01-01 10:00:00|14                |
|JFK    |2025-01-01 11:00:00|10                |
|JFK    |2025-01-01 12:00:00|15                |
|JFK    |2025-01-01 13:00:00|11                |
|JFK    |2025-01-01 14:00:00|23                |
|JFK    |2025-01-01 15:00:00|18                |
|JFK    |2025-01-01 16:00:00|24                |
|JFK    |2025-01-01 17:00:00|10                |
|JFK    |2025-01-01 18:00:00|23                |
|JFK    |2025-01-01 19:00:00|15                |
|JFK    |2025-01-01 

In [28]:
# Verify that aggregation preserves the total number of retained flights
hourly_arrival_total = (
    hourly_flight_arrivals
    .agg(F.sum("scheduled_arrivals").alias("total_scheduled_arrivals"))
)

hourly_arrival_total.show()
print("Expected retained flights:", scheduled_arrivals.count())

+------------------------+
|total_scheduled_arrivals|
+------------------------+
|                  117492|
+------------------------+

Expected retained flights: 117492


In [29]:
# Create a timezone-independent complete hourly grid
date_grid = (
    spark.range(1)
    .select(
        F.explode(
            F.sequence(
                F.to_date(F.lit("2025-01-01")),
                F.to_date(F.lit("2025-06-30")),
                F.expr("INTERVAL 1 DAY")
            )
        ).alias("date")
    )
)

clock_hour_grid = spark.range(24).select(
    F.col("id").cast("int").alias("hour")
)

hour_grid = (
    date_grid
    .crossJoin(clock_hour_grid)
    .withColumn(
        "arrival_hour",
        F.concat(
            F.date_format("date", "yyyy-MM-dd"),
            F.lit(" "),
            F.lpad(F.col("hour").cast("string"), 2, "0"),
            F.lit(":00:00")
        ).cast("timestamp_ntz")
    )
    .select("arrival_hour")
    .orderBy("arrival_hour")
)

print("Date rows:", date_grid.count())
print("Hours per day:", clock_hour_grid.count())
print("Hourly rows:", hour_grid.count())

Date rows: 181
Hours per day: 24
Hourly rows: 4344


In [30]:
airport_grid = spark.createDataFrame(
    [("JFK",), ("LGA",)],
    ["airport"]
)

complete_flight_panel = airport_grid.crossJoin(hour_grid)

print("Complete airport-hour panel rows:", complete_flight_panel.count())

Complete airport-hour panel rows: 8688


In [31]:
# Join observed scheduled arrivals onto the complete airport-hour panel
hourly_flight_complete = (
    complete_flight_panel
    .join(
        hourly_flight_arrivals,
        on=["airport", "arrival_hour"],
        how="left"
    )
    .fillna({"scheduled_arrivals": 0})
    .orderBy("airport", "arrival_hour")
)

print("Final flight panel rows:", hourly_flight_complete.count())

hourly_flight_complete.groupBy("airport").count().orderBy("airport").show()

Final flight panel rows: 8688
+-------+-----+
|airport|count|
+-------+-----+
|    JFK| 4344|
|    LGA| 4344|
+-------+-----+



In [32]:
# 1. Validate final time coverage
hourly_flight_complete.select(
    F.min("arrival_hour").alias("min_arrival_hour"),
    F.max("arrival_hour").alias("max_arrival_hour")
).show(truncate=False)

+-------------------+-------------------+
|min_arrival_hour   |max_arrival_hour   |
+-------------------+-------------------+
|2025-01-01 00:00:00|2025-06-30 23:00:00|
+-------------------+-------------------+



In [33]:
# 2. Count airport-hours with zero scheduled arrivals
hourly_flight_complete.groupBy("airport").agg(
    F.sum(
        (F.col("scheduled_arrivals") == 0).cast("int")
    ).alias("zero_arrival_hours")
).orderBy("airport").show()

+-------+------------------+
|airport|zero_arrival_hours|
+-------+------------------+
|    JFK|               464|
|    LGA|              1266|
+-------+------------------+



In [34]:
# 3. Summarise hourly scheduled arrivals by airport
hourly_flight_complete.groupBy("airport").agg(
    F.round(F.mean("scheduled_arrivals"), 2).alias("mean_arrivals"),
    F.min("scheduled_arrivals").alias("min_arrivals"),
    F.expr(
        "percentile_approx(scheduled_arrivals, 0.5)"
    ).alias("median_arrivals"),
    F.max("scheduled_arrivals").alias("max_arrivals")
).orderBy("airport").show()

+-------+-------------+------------+---------------+------------+
|airport|mean_arrivals|min_arrivals|median_arrivals|max_arrivals|
+-------+-------------+------------+---------------+------------+
|    JFK|        11.83|           0|             13|          28|
|    LGA|        15.22|           0|             19|          31|
+-------+-------------+------------+---------------+------------+



In [35]:
FLIGHT_OUTPUT_PATH = "../data/processed/hourly_flight_arrivals.parquet"

(
    hourly_flight_complete
    .write
    .mode("overwrite")
    .parquet(FLIGHT_OUTPUT_PATH)
)

saved_flight_arrivals = spark.read.parquet(FLIGHT_OUTPUT_PATH)

print("Saved rows:", saved_flight_arrivals.count())
print("Saved columns:", len(saved_flight_arrivals.columns))

saved_flight_arrivals.groupBy("airport").count().orderBy("airport").show()

print("Saved processed flight-arrival data to:")
print(FLIGHT_OUTPUT_PATH)

Saved rows: 8688
Saved columns: 3
+-------+-----+
|airport|count|
+-------+-----+
|    JFK| 4344|
|    LGA| 4344|
+-------+-----+

Saved processed flight-arrival data to:
../data/processed/hourly_flight_arrivals.parquet
